In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import plotnine as p9
import liana as li
import muon as mu
import anndata as ad
import matplotlib.pyplot as plt
import squidpy as sq
from liana.method._pipe_utils._common import _get_props
from liana.method.sp._utils import _add_complexes_to_var
from liana._logging import _logg
from liana.method._pipe_utils import prep_check_adata, assert_covered
from scipy.sparse import csr_matrix
from pathlib import Path
from plotnine import scale_y_continuous
from plotnine import geom_vline, annotate


/home/alsayah/.local/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
/home/alsayah/.local/lib/python3.10/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
/home/alsayah/.local/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.


# Load and preprocess

In [3]:
merfish_path = Path("/g/stegle/aalsayah/repos/data/merfish")
adata_main = sc.read(merfish_path / "mouse_brain/WB_MERFISH_animal2_coronal.h5ad")
adata = adata_main[adata_main.obs["brain_section_label"] == "C57BL6J-2.034"]
# filter cells and genes
sc.pp.filter_cells(adata, min_genes=10)
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

/home/alsayah/.local/lib/python3.10/site-packages/scanpy/preprocessing/_simple.py:176: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.


In [4]:
li.ut.spatial_neighbors(adata=adata, bandwidth=27, spatial_key="X_spatial_coords")
sq.gr.spatial_autocorr(adata, mode='moran', use_raw=False, n_jobs = 10,
                       show_progress_bar=True)
svgs = adata.uns['moranI'].index[(adata.uns['moranI']['pval_norm_fdr_bh'] < 0.05) & (adata.uns['moranI']['I'] > 0.01)]
adata = adata[:, svgs]

In [5]:
import mygene
mg = mygene.MyGeneInfo()
query = mg.querymany(adata.var_names.tolist(),
                     scopes="ensembl.gene",
                     fields="symbol",
                     species="mouse")

# Make a mapping dictionary {ensembl_id: symbol}
mapping = {q['query']: q.get('symbol', q['query']) for q in query}
adata.var_names = [mapping.get(gid, gid) for gid in adata.var_names]

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
querying 1-1000 ...
HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
querying 1001-1028 ...
HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
Finished.
2 input query terms found no hit:	['ENSMUSG00000113186', 'ENSMUSG00000109473']
Pass "returnall=True" to return complete lists of duplicate or missing query terms.


In [6]:
resource = li.rs.select_resource('consensus')
map_df = li.rs.get_hcop_orthologs(url='https://ftp.ebi.ac.uk/pub/databases/genenames/hcop/human_mouse_hcop_fifteen_column.txt.gz',
                                  columns=['human_symbol', 'mouse_symbol'],
                                   # NOTE: HCOP integrates multiple resource, so we can filter out mappings in at least 3 of them for confidence
                                   min_evidence=3
                                   )
map_df = map_df.rename(columns={'human_symbol':'source', 'mouse_symbol':'target'})
# We will then translate
mouse = li.rs.translate_resource(resource,
                                 map_df=map_df,
                                 columns=['ligand', 'receptor'],
                                 replace=True,
                                 # Here, we will be harsher and only keep mappings that don't map to more than 1 mouse gene
                                 one_to_many=1
                                 )
mouse
mouse.tuples = list(zip(mouse["ligand"], mouse["receptor"]))

/g/stegle/aalsayah/repos/liana-py/liana/resource/_orthology.py:199: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
/scratch/jobs/42213533/ipykernel_587775/2507512555.py:17: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access


In [7]:
lrdata = li.mt.inflow(adata,  
                                  groupby='cell_type', # cell type columns        
                                  interactions=mouse.tuples,                          
                                  use_raw=False)
                                  
lrdata.shape

/home/alsayah/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.


(49111, 1576)

# Target/Niche global score?


In [44]:
lrdata.obs["group_labels"] = lrdata.obs["cell_type"].astype(str) + "::" + lrdata.obs["major_brain_region"].astype(str)
li.mt.compute_global_score(lrdata=lrdata, groupby= "group_labels")

Running 500 permutations in parallel...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    1.3s
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:    7.1s
[Parallel(n_jobs=-1)]: Done 154 tasks      | elapsed:   16.9s
[Parallel(n_jobs=-1)]: Done 280 tasks      | elapsed:   30.2s
[Parallel(n_jobs=-1)]: Done 442 tasks      | elapsed:   47.6s
[Parallel(n_jobs=-1)]: Done 500 out of 500 | elapsed:   53.6s finished


In [45]:
lrdata.uns["global_score"][["target", "niche"]] =lrdata.uns["global_score"]["target"].str.split("::", expand=True)

In [46]:
cols = [c for c in lrdata.uns["global_score"].columns if c not in ["lr_mean", "pval"]]
cols += ["lr_mean", "pval"]
lrdata.uns["global_score"] = lrdata.uns["global_score"][cols]

In [47]:
lrdata.uns["global_score"].sort_values("pval")

,ligand,ligand_complex,receptor,receptor_complex,source,target,niche,lr_mean,pval
205201,Ccn2,Ccn2,Erbb4,Erbb4,astrocyte,oligodendrocyte,Ventricular_systems,0.051516,0.001996
25472,Fgf13,Fgf13,Egfr,Egfr,endothelial cell,astrocyte,Olfactory,0.177282,0.001996
90722,Wnt5a,Wnt5a,Epha7,Epha7,GABAergic neuron,glutamatergic neuron,Cortical_subplate,0.246811,0.001996
90719,Fgf13,Fgf13,Scn5a,Scn5a,GABAergic neuron,glutamatergic neuron,Cortical_subplate,0.534905,0.001996
90713,Penk,Penk,Adra2a,Adra2a,GABAergic neuron,glutamatergic neuron,Cortical_subplate,0.580221,0.001996
...,...,...,...,...,...,...,...,...,...
3,Rspo1,Rspo1,Lgr5,Lgr5,neuroblast (sensu Vertebrata),GABAergic neuron,Cortical_subplate,0.000000,1.000000
2,Thbs4,Thbs4,Cd36,Cd36,neuroblast (sensu Vertebrata),GABAergic neuron,Cortical_subplate,0.000000,1.000000
1,Hapln1,Hapln1,Tnfrsf11b,Tnfrsf11b,neuroblast (sensu Vertebrata),GABAergic neuron,Cortical_subplate,0.000000,1.000000
267919,Dcn,Dcn,Met,Met,vascular leptomeningeal cell,vascular leptomeningeal cell,nan,0.000000,1.000000
